# 05 — Cross-validation & Regularization
Bandingkan model menggunakan cross-validation dan Ridge.

In [1]:
from pathlib import Path
import pandas as pd
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
games = pd.read_csv(PROJECT_DIR / 'data' / 'game_sales_regression.csv')
features = ['rating','price_usd','review_count','marketing_budget_musd','release_year']
X = games[features]
y = games['sales_millions']
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import KFold, cross_val_score
cv=KFold(n_splits=5,shuffle=True,random_state=42)
models={'Linear':make_pipeline(StandardScaler(),LinearRegression()),'Polynomial':make_pipeline(StandardScaler(),PolynomialFeatures(degree=2,include_bias=False),LinearRegression()),'Polynomial + Ridge':make_pipeline(StandardScaler(),PolynomialFeatures(degree=2,include_bias=False),Ridge(alpha=1.0))}
for name,m in models.items():
 scores=-cross_val_score(m,X,y,cv=cv,scoring='neg_mean_absolute_error')
 print(f'{name}: CV MAE {scores.mean():.3f} +/- {scores.std():.3f}')

Linear: CV MAE 2.095 +/- 0.291
Polynomial: CV MAE 2.143 +/- 0.180
Polynomial + Ridge: CV MAE 2.140 +/- 0.181


## Eksperimen alpha

In [2]:
for alpha in [0.01,0.1,1,10,100]:
 m=make_pipeline(StandardScaler(),PolynomialFeatures(degree=2,include_bias=False),Ridge(alpha=alpha))
 scores=-cross_val_score(m,X,y,cv=cv,scoring='neg_mean_absolute_error')
 print(f'alpha={alpha}: MAE={scores.mean():.3f} +/- {scores.std():.3f}')

alpha=0.01: MAE=2.143 +/- 0.180
alpha=0.1: MAE=2.142 +/- 0.181
alpha=1: MAE=2.140 +/- 0.181
alpha=10: MAE=2.120 +/- 0.191
alpha=100: MAE=2.123 +/- 0.235


## Interpretasi
CV MAE lebih kecil berarti kesalahan validasi lebih rendah. Regularisasi terlalu kuat dapat menyebabkan underfitting.